# Explicit Table Transformation

This notebook applies reviewed transformation choices to copies of the raw tables. Extraction remains untouched, and validation remains observational: **Extraction → Validation → Transformation**.

## 1. Project setup and imports

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

working_directory = Path.cwd()
project_root = (
    working_directory
    if (working_directory / "app").exists()
    else working_directory.parent
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

sample_documents = project_root / "sample_documents"

from app.export.csv_exporter import export_dataframe_to_csv
from app.extraction.table_extractor import extract_table_as_dataframe
from app.parsers.pdf_parser import search_pdf
from app.transformation import transformer as transformer_module
from app.transformation.region_mapping import STATE_TO_REGION
from app.validation.table_validator import validate_table

transformer_module = importlib.reload(transformer_module)
assign_values_from_mapping = transformer_module.assign_values_from_mapping
find_unmapped_key_values = transformer_module.find_unmapped_key_values
transform_table = transformer_module.transform_table

print("Project root:", project_root)

Project root: C:\Users\vinee\document-table-agent


## 2. Locate and extract both raw tables

In [2]:
pdf_files = sorted(sample_documents.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError("No PDF files found in sample_documents.")

pdf_path = pdf_files[0]
energy_pages = search_pdf(pdf_path, "Energy Consumption")
maximum_pages = search_pdf(pdf_path, "Maximum Demand Met")

if not energy_pages or not maximum_pages:
    raise ValueError("One or both expected tables were not found.")

raw_energy_df = extract_table_as_dataframe(
    pdf_path, energy_pages[0], table_index=0
)
raw_maximum_df = extract_table_as_dataframe(
    pdf_path, maximum_pages[0], table_index=0
)

print("Energy raw shape:", raw_energy_df.shape)
print("Maximum Demand raw shape:", raw_maximum_df.shape)

Energy raw shape: (42, 9)
Maximum Demand raw shape: (42, 16)


## 3. Review validation hints before transforming

Header candidates are checked explicitly. They are not applied silently inside the transformation module.

In [3]:
energy_validation = validate_table(raw_energy_df)
maximum_validation = validate_table(raw_maximum_df)

assert energy_validation["possible_header_rows"] == [1]
assert maximum_validation["possible_header_rows"] == [1, 2]

display(
    pd.DataFrame(
        {
            "Energy Consumption": {
                "title rows": energy_validation["possible_title_rows"],
                "header rows": energy_validation["possible_header_rows"],
                "hierarchical": energy_validation["possible_hierarchical_header"],
            },
            "Maximum Demand": {
                "title rows": maximum_validation["possible_title_rows"],
                "header rows": maximum_validation["possible_header_rows"],
                "hierarchical": maximum_validation["possible_hierarchical_header"],
            },
        }
    )
)

,Energy Consumption,Maximum Demand
title rows,[0],[0]
header rows,[1],"[1, 2]"
hierarchical,False,True


## 4. Transform Energy Consumption

The reviewed single header is promoted, the seven measure columns are converted to numeric values, and Devanagari text is removed from the transformed copy. Region cells are intentionally not filled automatically.

In [4]:
energy_before_transformation = raw_energy_df.copy(deep=True)
transformed_energy_df = transform_table(
    raw_energy_df,
    header_row_positions=[1],
    numeric_column_positions=range(2, raw_energy_df.shape[1]),
    remove_devanagari=True,
)
pd.testing.assert_frame_equal(
    raw_energy_df,
    energy_before_transformation,
    check_exact=True,
)

print("Transformed Energy shape:", transformed_energy_df.shape)
display(transformed_energy_df.head())
display(transformed_energy_df.dtypes.to_frame("dtype"))

Transformed Energy shape: (40, 9)


,Region,States,30-03-2026,31-03-2026,01-04-2026,02-04-2026,03-04-2026,04-04-2026,05-04-2026
0,NR,Punjab,154.8,150.2,151.3,157.1,158.0,154.2,140.1
1,NaN,Haryana,158.5,154.2,150.2,158.9,155.8,153.1,132.6
2,NaN,Rajasthan,266.3,240.2,239.6,246.9,233.3,226.0,222.2
3,NaN,Delhi,93.0,84.4,88.9,93.6,93.4,88.4,85.5
4,NaN,UP,422.6,423.3,395.5,429.2,411.5,373.3,344.2


,dtype
Region,str
States,str
30-03-2026,float64
31-03-2026,float64
01-04-2026,float64
02-04-2026,float64
03-04-2026,float64
04-04-2026,float64
05-04-2026,float64


## 5. Transform Maximum Demand / Peak Shortage

The two reviewed header rows become a two-level `MultiIndex`. Date labels that came from horizontally merged cells are filled only across measure columns 2 onward, and Devanagari text is removed from the transformed copy. The hierarchical header is not flattened.

In [5]:
maximum_before_transformation = raw_maximum_df.copy(deep=True)
transformed_maximum_df = transform_table(
    raw_maximum_df,
    header_row_positions=[1, 2],
    fill_merged_headers_from_column=2,
    numeric_column_positions=range(2, raw_maximum_df.shape[1]),
    remove_devanagari=True,
)
pd.testing.assert_frame_equal(
    raw_maximum_df,
    maximum_before_transformation,
    check_exact=True,
)

print("Transformed Maximum Demand shape:", transformed_maximum_df.shape)
print("Column levels:", transformed_maximum_df.columns.nlevels)
display(transformed_maximum_df.head())
display(transformed_maximum_df.dtypes.to_frame("dtype"))

Transformed Maximum Demand shape: (39, 16)
Column levels: 2


Region       Date                     30-03-2026                   \
             States Max. Demand Met during the day Peak hr Shortage   
0            Punjab                           7517                0   
1     NR    Haryana                           7940                0   
2    NaN  Rajasthan                          12962                0   
3    NaN      Delhi                           4456                0   
4    NaN         UP                          21561                0   

                      31-03-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7441                0   
1                           7225                0   
2                          11326                0   
3                           4375                0   
4                          21301                0   

                      01-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7540                0   
1                           7944                0   
2                          11826                0   
3                           4367                0   
4                          22168                0   

                      02-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7579                0   
1                           8489                0   
2                          12382                0   
3                           4597                0   
4                          22797                0   

                      03-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7875                0   
1                           7529                0   
2                          11298                0   
3                           4445                0   
4                          21509                0   

                      04-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7277                0   
1                           8302                0   
2                          11019                0   
3                           4244                0   
4                          19134                0   

                      05-04-2026                   
  Max. Demand Met during the day Peak hr Shortage  
0                           6607                0  
1                           7166                0  
2                          10966                0  
3                           3961                0  
4                          19252                0

dtype
Region                                       str
Date       States                            str
30-03-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64
31-03-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64
01-04-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64
02-04-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64
03-04-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64
04-04-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64
05-04-2026 Max. Demand Met during the day  int64
           Peak hr Shortage                int64

## 6. Reconstruct Region from the reviewed mapping

Every non-total state/entity must exist in the explicit `STATE_TO_REGION` mapping. Unknown labels stop the pipeline instead of receiving a guessed region.

In [6]:
energy_unmapped = find_unmapped_key_values(
    transformed_energy_df, 1, STATE_TO_REGION
)
maximum_unmapped = find_unmapped_key_values(
    transformed_maximum_df, 1, STATE_TO_REGION
)
assert energy_unmapped == []
assert maximum_unmapped == []

transformed_energy_df = assign_values_from_mapping(
    transformed_energy_df,
    key_column_position=1,
    target_column_position=0,
    mapping=STATE_TO_REGION,
)
transformed_maximum_df = assign_values_from_mapping(
    transformed_maximum_df,
    key_column_position=1,
    target_column_position=0,
    mapping=STATE_TO_REGION,
)

region_counts = pd.concat(
    [
        transformed_energy_df.iloc[:, 0].value_counts().rename("Energy"),
        transformed_maximum_df.iloc[:, 0].value_counts().rename("Maximum Demand"),
    ],
    axis=1,
).fillna(0).astype(int)
display(region_counts)

,Energy,Maximum Demand
NR,10,10
WR,9,9
ER,7,7
NER,7,7
SR,6,6
ALL INDIA,1,0


## 7. Compare raw and transformed structures

In [7]:
comparison = pd.DataFrame(
    [
        {
            "table": "Energy Consumption",
            "raw shape": raw_energy_df.shape,
            "transformed shape": transformed_energy_df.shape,
            "column levels": transformed_energy_df.columns.nlevels,
            "numeric measure columns": 7,
            "Devanagari removed": True,
            "region mapping applied": True,
        },
        {
            "table": "Maximum Demand / Peak Shortage",
            "raw shape": raw_maximum_df.shape,
            "transformed shape": transformed_maximum_df.shape,
            "column levels": transformed_maximum_df.columns.nlevels,
            "numeric measure columns": 14,
            "Devanagari removed": True,
            "region mapping applied": True,
        },
    ]
)
display(comparison)

,table,raw shape,transformed shape,column levels,numeric measure columns,Devanagari removed,region mapping applied
0,Energy Consumption,"(42, 9)","(40, 9)",1,7,True,True
1,Maximum Demand / Peak Shortage,"(42, 16)","(39, 16)",2,14,True,True


## 8. Why mapping is used instead of forward-fill

Some vertically merged Region labels are emitted on a middle row rather than the first row of their visual group. Blindly forward-filling would assign preceding state rows to the wrong region. The reviewed state/entity mapping reconstructs Region deterministically and fails if a new label is not mapped.

## 9. Export clean transformed datasets

In [8]:
outputs = project_root / "outputs"
energy_output_path = export_dataframe_to_csv(
    transformed_energy_df,
    outputs / "clean_energy_consumption.csv",
    overwrite=True,
)
maximum_output_path = export_dataframe_to_csv(
    transformed_maximum_df,
    outputs / "clean_maximum_demand.csv",
    overwrite=True,
)

print("Saved:", energy_output_path)
print("Saved:", maximum_output_path)

Saved: C:\Users\vinee\document-table-agent\outputs\clean_energy_consumption.csv
Saved: C:\Users\vinee\document-table-agent\outputs\clean_maximum_demand.csv


## 10. Findings

- Both transformations operate on deep copies; assertions confirm that raw extraction is unchanged.
- The raw tables preserve their bilingual source text; only transformed copies remove Devanagari characters.
- English text and numeric values are retained, while leftover whitespace and empty delimiters are cleaned.
- Energy measure columns are numeric and its English-only reviewed header is promoted.
- Maximum Demand retains an English-only two-level date/measure header instead of flattening it.
- Region is reconstructed from a reviewed 39-entry state/entity mapping; no positional forward-fill is used.
- The Energy `ALL INDIA` row retains its existing total label.
- Clean CSV exports are written to `outputs/clean_energy_consumption.csv` and `outputs/clean_maximum_demand.csv`.
- No table merge or long-format reshape is performed in this milestone.